<center>
<h1 style="font-family:verdana">
 🏘 Sistema de diàleg basat en regles 🏘 <h1>








<p>   🎯 <b>Objectiu</b>: en aquesta pràctica aprendrem a crear un assistent senzill de compra d'habitatges. El sistema haurà d'identificar i mostrar les cases que coincideixen amb les preferències de l'usuari. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema siga més complet. </p>

<p> ✨ <b>Contingut</b>: en primer lloc, començarem amb un exemple senzill en què el sistema llançarà unes preguntes i respostes predefinides i l'usuari haurà d'escollir. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema siga més complet. </p>

✏ <b>Exercicis</b>: en cada secció anireu trobant exercicis que haureu d'anar resolent.

---

<h2> Índex </h2>


1. [Fitxer JSON](#section-one)
  * [Exercici 1](#ex-one)
2. [Sistema de diàleg senzill](#section-two)
  * [Exercici 2](#ex-two)
  * [Exercici 3](#ex-three)
  * [Exercici 4](#ex-four)
3. [Millorem el sistema de diàleg](#section-three)
  * [Exercici 5](#ex-five)
  * [Exercici 6](#ex-six)
4. [Lliurable](#section-four)
---


In [1]:
from nltk.tokenize.treebank import TreebankWordTokenizer
import nltk
# nltk.download('omw-1.4')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')
# nltk.download('punkt_tab')
# !pip install text2num   # números en paraules ("twenty-one", "three thousand"...)

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

import json
import sys
import difflib

from text_to_num import alpha2digit

<h1><a name="section-one"> 1. Fitxer JSON </a></h1>

En primer lloc, carregarem el fitxer JSON per analitzar-ne l'estructura.

In [2]:
with open('house_data.json') as f:
  data = json.load(f)

print('Start message: ', data['start_message'])
print('End message: ', data['end_message'])
print('Questions: ', data['questions'])
print('House: ', data['houses'])

Start message:  Welcome to the House Buying Assistant!
End message:  Thank you for using the House Buying Assistant. Goodbye!
Questions:  [{'question': 'How many bedrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 5 bedrooms): ', 'answer_key': 'bedrooms'}, {'question': 'How many bathrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 3 bathrooms): ', 'answer_key': 'bathrooms'}, {'question': 'What is your budget for the house?', 'type': 'numerical', 'prompt': 'Enter your choice (1k - 999k euros):', 'answer_key': 'price'}, {'question': 'How many square meters do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (30 - 200 meters)', 'answer_key': 'square_meters'}, {'question': 'Which city or neighborhood would you prefer?', 'type': 'multichoice', 'prompt': 'Enter your choice', 'answer_key': 'location'}, {'question': 'Are you looking to buy or rent a house?', 'type': 'multichoice', 'prompt': 'Enter your choice (sale or rent): ',

Podeu veure com al fitxer JSON podem trobar el missatge de benvinguda i el missatge de comiat. Les diferents preguntes predefinides que el sistema demanarà a l'usuari. I les cases que l'agència té disponibles per a oferir a l'usuari.

Si no acabeu d'entendre el format, podeu copiar el contingut del fitxer JSON [aquí](https://jsonviewer.stack.hu/) per visualitzar millor l'estructura.

 <h1><a name="ex-seven"><center> ✏ Exercici 7 ✏ </a></h1>

Ara ja teniu un sistema funcional. Aplicant el mètode vist a classe (Exemples, Camins, Prototipat i Proves) milloreu aquest sistema per donar una millor experiència als usuaris.

---

## 7.1 Interfície del xat

Aquesta cel·la crea la interfície amb `ipywidgets`: una caixa de text, un botó d'enviar i un registre de conversa (`chat_history`) que es pinta com a bombolles. `bubble_agent` mostra els missatges del sistema, `bubble_user` els de l'usuari i `debug_message` mostra missatges de depuració quan el `debug` està activat. Aquesta part només s'encarrega de l'aspecte visual; la lògica del diàleg és a les cel·les següents.

In [3]:
import html
import ipywidgets as widgets
from IPython.display import display

chat_history = []
CHAT_BOX_HEIGHT = 260  # px

# Ja NO posem height/overflow_y al layout: només al contenidor exterior
chat_log = widgets.HTML()

text_box = widgets.Text(
    placeholder="Type your answer and press Enter...",
    layout=widgets.Layout(width="80%")
)

send_btn = widgets.Button(description="Send", button_style="primary")

chat_ui = widgets.VBox([chat_log, widgets.HBox([text_box, send_btn])])


def render_chat():
    # El scroll i l'alçada van al propi HTML, amb CSS inline
    scroll_hack = (
        '<img src="x" style="display:none" '
        'onerror="var el=document.getElementById(\'chat-scroll-box\'); '
        'if(el){el.scrollTop = el.scrollHeight;}">'
    )
    chat_log.value = f'''
        <div id="chat-scroll-box" style="height:{CHAT_BOX_HEIGHT}px;
             overflow-y:auto; border:1px solid #ddd; padding:8px;
             box-sizing:border-box;">
            {"".join(chat_history)}
        </div>
        {scroll_hack}
    '''


def bubble_agent(message):
    safe_message = html.escape(str(message))
    chat_history.append(f"""
        <div style="text-align:left;margin:6px 0;font-family:sans-serif;">
            <span style="background:#eef2ff;padding:8px 14px;
            border-radius:14px 14px 14px 2px;display:inline-block;
            max-width:75%;font-size:14px;white-space:pre-line;">🏠 {safe_message}</span>
        </div>
    """)
    render_chat()


def bubble_user(message):
    safe_message = html.escape(str(message))
    chat_history.append(f"""
        <div style="text-align:right;margin:6px 0;font-family:sans-serif;">
            <span style="background:#dcfce7;padding:8px 14px;
            border-radius:14px 14px 2px 14px;display:inline-block;
            max-width:75%;font-size:14px;">{safe_message} 🧑</span>
        </div>
    """)
    render_chat()

def debug_message(message):
    safe_message = html.escape(str(message))

    chat_history.append(f"""
        <div style="text-align:left;margin:6px 0;font-family:sans-serif;">
            <span style="background:#fff7ed;color:#9a3412;padding:8px 14px;
            border-radius:8px;display:inline-block;max-width:90%;
            font-size:13px;">
            DEBUG: {safe_message}
            </span>
        </div>
    """)

    render_chat()

## 7.2 Dades i preguntes

Reutilitzem `data['questions']` (bedrooms, bathrooms, price, square_meters, location) i afegim, des del codi i no des del JSON, les preguntes de l'exercici 6: `type`, `monthly_income`, `floor`, `terrace`, `elevator`, `commercial_use`.

## 7.3 Resum de millores de l'exercici 7

Sobre el prototip anterior, s'ha afegit:

- **Sinònims + negació** a les preguntes Sí/No (terrace, elevator, commercial_use): "yeah", "sure", "I don't want a terrace"...
- **Números en paraules** amb la llibreria `text2num` ("twenty-one", "one hundred and fifty", "three thousand", "fifth floor"...), amb un diccionari de reserva (`WORD_NUMBERS`) per a "couple", "single", "ground"... si la llibreria no està instal·lada.
- **Sinònims per a "any"** ("whatever", "I don't care", "no preference", "up to you"...), distingint paraules soltes (per token, evitant falsos positius com "many") de frases de 2+ paraules (per substring, on el risc de xoc és molt menor).
- **Confirmació curta** després de cada resposta ("Got it — 3 bedrooms."), amb 2-3 variants triades a l'atzar, igual que les preguntes.
- **Resum** de totes les preferències just abans de cercar ("Renting, 3 bedrooms, Barcelona, up to 1200€ — searching...").
- **`back`** per tornar a la pregunta anterior sense reiniciar tota la conversa (es guarda un historial de snapshots). Ara es reconeix també dins d'una frase ("I want to go to the previous question") o quan l'usuari diu que s'ha equivocat ("I made a mistake", "wrong answer"...); en tots dos casos el sistema **demana confirmació** abans de tornar enrere.
- **Flux condicional de pressupost**: si `type = sale` es pregunta directament el preu màxim; si `type = rent` primer es pregunta si es vol calcular el pressupost a partir del salari (regla del 35%) o si ja es té un preu màxim en ment, i només es fa **una** de les dues preguntes (income o price) segons la resposta.
- Si `commercial_use = Yes`, el sistema **recomana planta baixa i pregunta si hi està d'acord**: si diu que sí, s'aplica el filtre de planta 0 i **ja no es pregunta la planta**; si diu que no, es pregunta la planta mínima amb normalitat (hi ha locals comercials en altres plantes). La pregunta de la **planta mínima** va just després del bloc d'ús comercial, de manera que totes les branques (`commercial_use = No`, o `Yes` amb planta baixa rebutjada) conflueixen en la mateixa pregunta i el flux no es duplica.
- **Marge de metres quadrats asimètric**: es toleren cases una mica més grans del que es demana (`SQM_MARGIN_ABOVE = 25`) més que més petites (`SQM_MARGIN_BELOW = 10`) — és preferible passar-se que quedar-se curt.
- **Pressupost com a topall estricte** a la cerca (sense marge, ni en compra ni en lloguer). Si l'usuari demana relaxar-lo, el topall puja un percentatge modest (`BUDGET_RELAXATION_PERCENT = 10%`) cada vegada. Funciona igual si el pressupost ve d'un preu directe o del càlcul per ingressos (regla del 35%).
- **Relaxar requisits (opcional)**: l'usuari ho demana quan vol (`relax`), des de qualsevol llista de cases, i tria **un requisit cada vegada** (pel número o per paraula clau: "the budget", "terrace"...). El menú mostra només els requisits actius i, al costat, **quantes cases més apareixerien** (les que només fallen en aquest camp, calculat de la mateixa llista de desajustos, sense tornar a cercar). Després el bot pregunta "Do you want to relax anything else?" i, quan l'usuari diu que no, es **torna a puntuar** amb les preferències noves. Bedrooms, bathrooms, location, m², planta mínima, terrassa, ascensor i planta baixa passen a `any` (el pressupost puja un 10%). `type` i `commercial_use = Yes` no es negocien mai.
- **Ús comercial**: només `Yes` restringeix la cerca (cases amb ús comercial, i planta baixa només si l'usuari ha acceptat la recomanació); un `No` ja no exclou cases que simplement el permeten.
- **Llistes d'una línia per casa** amb el format `House 3 · Santa Coloma de Gramenet · 800€/month`. El detall complet (`House 3 · ... — 2 bedrooms, ...`) només es mostra quan l'usuari tria una casa, i després **es torna a la mateixa llista**: es poden veure els detalls de diverses cases abans de decidir.
- **Cerca per puntuació (una sola funció)**: `score_houses()` calcula per a cada casa la llista de desajustos (`Mismatch`: camp, text i gravetat entre 0 i 1). Les cases amb **0 desajustos són les exactes** i es mostren totes; després es pregunta si li agrada alguna o prefereix veure **alternatives**: les 3 millors amb desajustos (ordenades per nombre de desajustos, després per gravetat total i després per id), amb `more` per veure les 3 següents. Si no hi ha cap casa exacta, el bot ho diu i passa directament a les alternatives. `type` i `commercial_use = Yes` són requisits durs; la planta baixa per a ús comercial també filtra, però es pot relaxar.
- **Respostes d'incertesa**: "I don't know", "not sure", "no idea"... ja **no** es llegeixen com un "No"; es guarden com a `any` i el sistema ho diu ("No worries, I'll mark it as 'any'").
- **Abreviatures**: `normalize_text()` s'aplica una sola vegada a l'entrada (`idk`, `idc`, `dunno`, `pls`, `bk`... i, només a les preguntes Sí/No, `y`, `n`, `ya`, `k`...). També uniforma els apòstrofs corbs (`’`).
- **Rangs numèrics** (`NUMERIC_RANGES`): si el valor està fora de rang o no és enter on toca, el sistema explica el rang vàlid en lloc d'acceptar-lo. També s'entenen separadors de milers ("1,200€", "1.200€") i s'ha corregit el cas "3 kitchens" (la `k` ja no es llegeix com a "mil").
- **Confirmació en sortir**: `q`/`quit`/`exit` pregunta "Are you sure you want to leave the chat?" en qualsevol fase (preguntes, llistes de resultats i relaxació).
- **Pressupost en lloguer amb tres opcions**: preu màxim, càlcul a partir dels ingressos, o **obert** (sense límit; es guarda com a `any` i no es pregunta ni ingressos ni preu). Frases com "I don't have a budget" es reconeixen com a obertes en lloc de com a "budget".
- **Organització del codi**: l'estat de la conversa (quina pregunta toca, què s'ha respost, l'historial per al `back`...) s'ha encapsulat en una classe `DialogEngine`, en lloc d'un diccionari global `dialog_state` mutat per funcions soltes. Les funcions que no depenen de cap estat (parsing de respostes, cerca de cases...) s'han deixat tal com estaven, com a funcions independents.


**Preguntes noves.** El JSON només té cinc preguntes (`bedrooms`, `bathrooms`, `price`, `square_meters`, `location`). Aquesta cel·la afegeix, des del codi, les que necessita el nostre flux: tipus d'operació (`type`), mètode de pressupost (`budget_method`), ingressos (`monthly_income`), planta mínima (`floor`), terrassa, ascensor, ús comercial i la proposta de planta baixa (`commercial_floor`). Cada pregunta té tres variants de text.

In [4]:
NEW_QUESTIONS = [
    {
        "question": "Are you looking to buy or rent a house?",
        "question_variants": [
            "Are you looking to buy or rent a house?",
            "Would you like to buy or rent?",
            "So, are we talking about buying or renting?",
        ],
        "type": "multichoice",
        "prompt": "Enter sale or rent: ",
        "answer_key": "type",
    },
    {
        "question": "How would you like to set your rent budget? I can calculate it from your income, you can give me a maximum price, or we can leave it open if you don't have a limit in mind.",
        "question_variants": [
            "How would you like to set your rent budget? I can calculate it from your income, you can give me a maximum price, or we can leave it open if you don't have a limit in mind.",
            "For the rent budget, should I work it out from your income, do you have a maximum price, or would you rather leave it open?",
            "Do you want me to estimate your budget from your income, set a price limit, or leave the budget open?",
        ],
        "type": "multichoice",
        "prompt": "Enter 'income', 'budget' or 'any': ",
        "answer_key": "budget_method",
    },
    {
        "question": "What is your household's monthly income?",
        "question_variants": [
            "What is your household's monthly income?",
            "How much does your household earn per month?",
            "What's your monthly income, roughly?",
        ],
        "type": "numerical",
        "prompt": "Enter your monthly income in euros: ",
        "answer_key": "monthly_income",
    },
    {
        "question": "What is the minimum floor you would like to live on?",
        "question_variants": [
            "What is the minimum floor you would like to live on?",
            "Is there a minimum floor you'd like to be on?",
            "How low or high can the floor be, at minimum?",
        ],
        "type": "numerical",
        "prompt": "Enter the minimum floor (0 or higher): ",
        "answer_key": "floor",
    },
    {
        "question": "Do you want a house with a terrace?",
        "question_variants": [
            "Do you want a house with a terrace?",
            "Is a terrace important to you?",
            "Would you like the house to have a terrace?",
        ],
        "type": "multichoice",
        "prompt": "Enter Yes or No: ",
        "answer_key": "terrace",
    },
    {
        "question": "Do you want a house with an elevator?",
        "question_variants": [
            "Do you want a house with an elevator?",
            "Do you need the building to have an elevator?",
            "Is having an elevator a must for you?",
        ],
        "type": "multichoice",
        "prompt": "Enter Yes or No: ",
        "answer_key": "elevator",
    },
    {
        "question": "Will you use the house for commercial purposes?",
        "question_variants": [
            "Will you use the house for commercial purposes?",
            "Are you planning to use it as a business or shop?",
            "Is this for commercial use?",
        ],
        "type": "multichoice",
        "prompt": "Enter Yes or No: ",
        "answer_key": "commercial_use",
    },
    {
        "question": "Since it's for commercial use, I'd recommend a ground-floor (floor 0) property. Do you agree?",
        "question_variants": [
            "Since it's for commercial use, I'd recommend a ground-floor (floor 0) property. Do you agree?",
            "For commercial use, ground floor (0) is usually the best fit. Does that work for you?",
            "I'd suggest a ground-floor property for a business. Are you okay with that?",
        ],
        "type": "multichoice",
        "prompt": "Enter Yes or No: ",
        "answer_key": "commercial_floor",
    },
]

existing_keys = {q["answer_key"] for q in data["questions"]}
for question in NEW_QUESTIONS:
    if question["answer_key"] not in existing_keys:
        data["questions"].append(question)
        existing_keys.add(question["answer_key"])

print([q["answer_key"] for q in data["questions"]])


['bedrooms', 'bathrooms', 'price', 'square_meters', 'location', 'type', 'monthly_income', 'floor', 'terrace', 'elevator', 'commercial_use', 'budget_method', 'commercial_floor']


**Variants de les preguntes originals.** Afegeix tres variants de text a les cinc preguntes que ja venien del JSON, perquè el sistema no repeteixi sempre la mateixa frase.

In [5]:
ORIGINAL_QUESTION_VARIANTS = {
    "bedrooms": [
        "How many bedrooms do you need?",
        "How many bedrooms are you looking for?",
        "What number of bedrooms would suit you?",
    ],
    "bathrooms": [
        "How many bathrooms do you need?",
        "How many bathrooms would you like?",
        "And bathrooms, how many do you need?",
    ],
    "price": [
        "What is the maximum you're willing to spend on the house?",
        "What's the top budget you'd go up to?",
        "How much would you spend at most on this house?",
    ],
    "square_meters": [
        "How many square meters do you need? (similar sizes are fine too)",
        "What size are you looking for, in square meters? A close size also works.",
        "How much space do you need, in m²? We'll consider similar sizes as well.",
    ],
    "location": [
        "Which city or neighborhood would you prefer?",
        "Where would you like to live?",
        "Do you have a city or neighborhood in mind?",
    ],
}

for question in data["questions"]:
    key = question["answer_key"]
    if key in ORIGINAL_QUESTION_VARIANTS and "question_variants" not in question:
        question["question_variants"] = ORIGINAL_QUESTION_VARIANTS[key]


## 7.4 Interpretació de les respostes: sinònims i normalització

Aquí es defineixen els diccionaris i les funcions que permeten entendre respostes lliures en lloc de paraules exactes: números en paraules, paraules afirmatives i negatives, abreviatures (`idk`, `pls`...), sinònims per a `type` i `budget_method`, frases per dir que no importa (`any`) o que no se sap, i la coincidència aproximada per a la ubicació (`fuzzy_match`). `normalize_text` s'aplica una sola vegada a cada entrada de l'usuari.

In [6]:
import re

# Sinonims i paraules que el sistema ha de reconeixer, a banda de la resposta "literal"

# Números en paraules (útil per a bedrooms/bathrooms, rangs petits 1-5 / 1-3,
# i també per a floor, que sol ser un número baix)
WORD_NUMBERS = {
    "zero": 0, "ground": 0,
    "one": 1, "single": 1, "a": 1,
    "two": 2, "couple": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
}

# Paraules que indiquen afirmació / negació per a les preguntes Sí/No
# (terrace, elevator, commercial_use). Es miren com a "bag of words" perquè
# funcionin encara que vinguin dins d'una frase ("yeah, I'd love a terrace").
AFFIRMATIVE_WORDS = {
    "yes", "yeah", "yep", "yup", "sure", "definitely", "affirmative",
    "please", "want", "wanna", "need", "like", "love", "ok", "okay",
    "correct", "right", "indeed", "certainly", "absolutely", "si", "sí",
}
NEGATIVE_WORDS = {
    "no", "nope", "nah", "not", "never", "n't", "dont", "don't", "doesnt",
    "doesn't", "without", "skip", "negative", "none",
}


# ---------------------------------------------------------------------
# Normalització del text d'entrada (es fa UNA sola vegada a handle_answer)
# ---------------------------------------------------------------------
# Abreviatures típiques de l'anglès que es poden usar en qualsevol pregunta
GENERAL_ABBREVIATIONS = {
    "idk": "i don't know", "dunno": "i don't know", "idc": "i don't care",
    "plz": "please", "pls": "please", "thx": "thanks",
    "bk": "back", "prev": "previous",
    "dont": "don't", "doesnt": "doesn't", "cant": "can't",
}

# Abreviatures només per a preguntes Sí/No ("y", "n", "k"... serien ambigües
# en una pregunta numèrica: "100 k" és 100.000, no un "ok")
YES_NO_ABBREVIATIONS = {
    "y": "yes", "ya": "yes", "yh": "yes", "ye": "yes", "yea": "yes", "yeh": "yes",
    "ofc": "yes", "k": "ok", "kk": "ok",
    "n": "no", "nop": "no",
}


def _expand_abbreviations(text, mapping):
    # Els lookarounds eviten tocar "n/a", "don't", "know"... (només paraules soltes)
    pattern = (r"(?<![\w/'])(" + "|".join(sorted(map(re.escape, mapping), key=len, reverse=True))
               + r")(?![\w/'])")
    return re.sub(pattern, lambda m: mapping[m.group(1)], text)


def normalize_text(text, yes_no=False):
    """Minúscules, apòstrofs normals, espais nets i abreviatures expandides."""
    text = str(text).lower().replace("’", "'").replace("‘", "'")
    text = " ".join(text.split())
    text = _expand_abbreviations(text, GENERAL_ABBREVIATIONS)
    if yes_no:
        text = _expand_abbreviations(text, YES_NO_ABBREVIATIONS)
    return text


# "No ho sé" -> es guarda com a 'any' (i es diu a l'usuari)
UNCERTAINTY_SINGLE_WORDS = {"unsure", "undecided", "dunno"}
UNCERTAINTY_PHRASES = {
    "don't know", "do not know", "dont know", "not sure", "no idea", "no clue",
    "not certain", "can't decide", "cant decide", "can't say", "hard to say",
    "haven't decided", "havent decided", "not decided", "not made up my mind",
}

# Pressupost "obert" (opció 'any' de budget_method). Es comprova ABANS dels
# sinònims perquè "I don't have a budget" conté la paraula "budget".
BUDGET_ANY_WORDS = {"open", "unlimited", "unrestricted", "anything"}
BUDGET_ANY_PHRASES = {
    "no budget", "no limit", "no max", "no maximum", "no cap", "no price",
    "without limit", "without a limit", "without a budget",
    "don't have a budget", "dont have a budget", "do not have a budget",
    "don't have a limit", "dont have a limit", "do not have a limit",
    "don't have any budget", "haven't got a budget", "no budget in mind",
    "no price limit", "not have a budget", "not have a limit",
}


def parse_yes_no(answer):
    """Interpreta una resposta lliure com Yes/No, tenint en compte negacions
    ("I don't want a terrace") i sinònims habituals ("yeah", "sure", "nah")."""
    text = normalize_text(answer, yes_no=True)
    if is_uncertain_answer(text):
        return None   # "I don't know" no és un "No": ho gestiona el flux principal
    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    if tokens & NEGATIVE_WORDS:
        return "No"
    if tokens & AFFIRMATIVE_WORDS:
        return "Yes"
    return None


# Camps que es tracten com preguntes Sí/No amb sinònims + negació
YES_NO_FIELDS = {"terrace", "elevator", "commercial_use", "commercial_floor"}

# Sinònims específics per a preguntes de tipus "multichoice" que no són Sí/No
SYNONYMS = {
    "type": {
        "sale": ["buy", "buying", "purchase", "purchasing", "sale", "sell", "own", "owning"],
        "rent": ["rent", "renting", "lease", "leasing", "rental"],
    },
    "budget_method": {
        "income": ["income", "salary", "salaries", "earnings", "wage", "wages", "calculate", "compute", "estimate"],
        "budget": ["budget", "price", "amount", "cap", "max", "maximum", "figure", "limit"],
    },
    # Aquests tres també es poden reconèixer per paraula clau (a més del Sí/No genèric)
    "terrace": {"Yes": ["terrace", "balcony"], "No": []},
    "elevator": {"Yes": ["elevator", "lift"], "No": []},
    "commercial_use": {"Yes": ["shop", "business", "store", "commercial", "office"], "No": []},
}

# Sinònims per a "no tinc preferència" (comanda 'any')
# - Paraules soltes ambigües -> es comproven per TOKEN sencer (evita falsos positius
#   com "any" dins de "many", o "not" ficant-se pel mig d'un "No" real)
# - Frases de 2+ paraules -> es comproven com a substring, el risc de xoc és molt menor
INDIFFERENCE_SINGLE_WORDS = {"any", "whatever", "indifferent", "flexible"}
INDIFFERENCE_PHRASES = {
    "no preference", "no particular preference", "no strong preference",
    "doesn't matter", "does not matter", "dont matter", "do not matter",
    "don't care", "dont care", "do not care",
    "not fussed", "not bothered", "not picky", "not particular",
    "either way", "either is fine", "either works",
    "you decide", "up to you", "i'm flexible", "im flexible",
}

FUZZY_FIELDS = {"location"}

def fuzzy_match(text, options, cutoff=0.6):
    text = text.strip().lower()
    options_lower = {opt.lower(): opt for opt in options}

    # Primer provem la frase sencera
    match = difflib.get_close_matches(text, options_lower.keys(), n=1, cutoff=cutoff)
    if match:
        return options_lower[match[0]]

    # Si no, provem paraula per paraula (per si escriuen "in Barcelon please")
    for token in text.split():
        match = difflib.get_close_matches(token, options_lower.keys(), n=1, cutoff=cutoff)
        if match:
            return options_lower[match[0]]

    return None


## 7.5 Anàlisi i validació de les respostes

Funcions que converteixen el text de l'usuari en un valor utilitzable:

- **Opcions disponibles**: es construeixen a partir de les cases del JSON (`initialize_available_options`).
- **Números**: `get_numerical_value` extreu un número del text (amb `€`, `k`, separadors de milers o escrit en paraules) i `validate_numeric_answer` el comprova contra els rangs de `NUMERIC_RANGES`.
- **Opcions de text**: `validate_multichoice_answer` prova, per ordre, coincidència exacta, Sí/No amb negació, pressupost obert, sinònims i coincidència aproximada.
- **Comandes**: detecció de sortida, `any`, incertesa, tornar enrere i comentaris de l'estil "m'he equivocat".

La darrera línia (`check_numeric_ranges_against_data`) imprimeix si els rangs numèrics encaixen amb les cases reals.

In [7]:
import re

def initialize_available_options(house_data, available_options):
    for house in house_data['houses']:
        for key, value in house.items():
            available_options.setdefault(key, set()).add(value)


available_options = {}
initialize_available_options(data, available_options)

# Opcions que no venen de les cases (perquè és una pregunta nova del sistema,
# no un camp de la casa) s'han de declarar a mà
CUSTOM_OPTIONS = {
    "budget_method": ["income", "budget", "any"],
    "commercial_floor": ["Yes", "No"],
}



# Ordinals que text2num deixa en paraules ("first", "second") + altres
ORDINAL_WORDS = {
    "first": 1,
    "second": 2,
    "third": 3,
    "fourth": 4,
    "fifth": 5,
    "sixth": 6,
    "seventh": 7,
    "eighth": 8,
    "ninth": 9,
    "tenth": 10,
}


def _strip_thousands_separators(text):
    """'1,200' / '1.200' / '1,200,000' -> 1200 / 1200 / 1200000.
    Una coma o punt seguit d'EXACTAMENT 3 dígits és un separador de milers;
    qualsevol altra coma es tracta com a decimal ('2,5' -> '2.5')."""
    text = re.sub(r"(?<=\d)[,.](?=\d{3}(?!\d))", "", text)
    return text.replace(",", ".")


def get_numerical_value(answer_text):
    """
    Busca un número al text original, admetent:
    - Enters i decimals (100, 100.5, 100,5) i separadors de milers (1,200 / 1.200)
    - Sufix 'k'/'K' enganxat o amb espai (100k, 100 k, 100K), però NO "3 kitchens"
    - Símbol € opcional, abans o després
    - Signe negatiu opcional al principi
    - Números escrits en paraules: "twenty-one", "one hundred and fifty",
      "three thousand", "fifth floor" (text2num) i "couple", "single", "ground"...
    Retorna el valor com a float, ja multiplicat si hi havia 'k'.
    """
    text = answer_text.lower().replace('€', '').replace('’', "'").strip()


    # threshold=0: converteix també "one", "two"... (per defecte es queden en paraules)
    text = alpha2digit(text, "en", threshold=0)

    text = _strip_thousands_separators(text)

    match = re.search(r'(-?\d+(?:\.\d+)?)(?:\s*(k)\b)?', text)
    if match:
        number = float(match.group(1))
        if match.group(2) == 'k':
            number *= 1000
        return number

    # Si no hi havia dígits, provem amb el diccionari de reserva. "a" va al
    # final: a "a couple of bedrooms" ha de guanyar "couple", no "a".
    tokens = nltk.word_tokenize(text)
    for token in tokens:
        if token in WORD_NUMBERS and token != "a":
            return float(WORD_NUMBERS[token])
    for token in tokens:
        if token in ORDINAL_WORDS:
            return float(ORDINAL_WORDS[token])
    if "a" in tokens:
        return float(WORD_NUMBERS["a"])

    return ''


def is_exit_command(answer):
    return answer.strip().lower() in {"q", "quit", "exit", "salir"}


def is_uncertain_answer(answer):
    """'I don't know', 'not sure', 'idk'... Si la resposta també conté un número
    ("I don't know, maybe 3") la tractem com una resposta normal."""
    text = normalize_text(answer)
    if re.search(r"\d", text):
        return False
    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    return bool(tokens & UNCERTAINTY_SINGLE_WORDS) or any(p in text for p in UNCERTAINTY_PHRASES)


def is_open_budget_answer(answer):
    """'no limit', 'I don't have a budget', 'open'... (opció 'any' de budget_method)."""
    text = normalize_text(answer)
    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    return bool(tokens & BUDGET_ANY_WORDS) or any(p in text for p in BUDGET_ANY_PHRASES)


def is_no_answer(answer):
    """Un 'no' per renunciar a la negociació / cases més properes ('no', 'nope', 'n'...)."""
    text = normalize_text(answer)
    if text == "no":
        return True
    return not re.search(r"\d", text) and parse_yes_no(text) == "No"


def is_any_command(answer):
    text = normalize_text(answer)
    if text == "any" or text == "n/a":
        return True

    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    if tokens & INDIFFERENCE_SINGLE_WORDS:
        return True

    return any(phrase in text for phrase in INDIFFERENCE_PHRASES)


# Tornar enrere: ara es detecta també dins d'una frase
# ("I want to go to the previous question", "can we go back?"...).
BACK_WORDS = {"back", "previous", "undo", "atras", "atrás", "enrere", "anterior"}
BACK_PHRASES = {"go back", "previous question", "last question", "question before", "step back"}

# Frases que indiquen que l'usuari creu que s'ha equivocat. Aquí NO tornem
# enrere directament: primer demanem confirmació, perquè és més ambigu.
MISTAKE_WORDS = {"mistake", "mistaken", "mistyped", "typo", "oops", "whoops",
                 "wrong", "misspoke", "misclicked"}
MISTAKE_PHRASES = {"didn't mean", "didnt mean", "did not mean", "not what i meant",
                   "change my answer", "change my mind", "made an error", "my bad",
                   "by mistake"}


def is_back_command(answer):
    text = answer.strip().lower()
    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    return bool(tokens & BACK_WORDS) or any(phrase in text for phrase in BACK_PHRASES)


def is_mistake_comment(answer):
    text = answer.strip().lower().replace("’", "'")
    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    return bool(tokens & MISTAKE_WORDS) or any(phrase in text for phrase in MISTAKE_PHRASES)


def numeric_value_from_text(value):
    try:
        return float(str(value).lower().replace('k', '000').replace('€', '').strip())
    except ValueError:
        return None


# Rangs vàlids per a les respostes numèriques (claus = answer_key de la pregunta).
# Els rangs de bedrooms/bathrooms/square_meters s'haurien d'ajustar a les cases
# reals: check_numeric_ranges_against_data() (al final d'aquesta cel·la) ho comprova.
NUMERIC_RANGES = {
    'bedrooms': (1, 5),
    'bathrooms': (1, 3),
    'square_meters': (30, 200),
    'monthly_income': (1, float('inf')),   # 0€ donaria un pressupost de 0€ (cap resultat)
    'price': (1, float('inf')),
    'floor': (0, 10),
}

# Només enters (no té sentit "2.5 habitacions")
INTEGER_FIELDS = {'bedrooms', 'bathrooms', 'floor'}

RANGE_LABELS = {
    'bedrooms': 'bedrooms',
    'bathrooms': 'bathrooms',
    'square_meters': 'square meters',
    'monthly_income': 'monthly income (€)',
    'price': 'maximum price (€)',
    'floor': 'floor',
}


def validate_numeric_answer(answer, answer_key=None):
    """Retorna (valor, None) si és vàlid, o (None, missatge_d'error) si no."""
    value = get_numerical_value(answer)
    if value == '':
        return None, ("Sorry, I didn't catch a number there. Try again, "
                      "or type 'any' if it doesn't matter to you.")

    label = RANGE_LABELS.get(answer_key, "that")
    low, high = NUMERIC_RANGES.get(answer_key, (0, float('inf')))

    if answer_key in INTEGER_FIELDS and value != int(value):
        return None, f"For {label}, please enter a whole number (or type 'any')."

    if value < low or value > high:
        if high == float('inf'):
            return None, f"For {label}, please enter {low:g} or more (or type 'any')."
        return None, f"For {label}, please enter a value between {low:g} and {high:g} (or type 'any')."

    return value, None


def validate_multichoice_answer(answer, options, answer_key=None):
    normalized = {opt.lower(): opt for opt in options}

    # 1. Coincidència exacta
    exact = normalized.get(answer.strip().lower())
    if exact is not None:
        return exact

    # 2. Preguntes Sí/No: sinònims + negació ("yeah", "I don't want a terrace"...)
    if answer_key in YES_NO_FIELDS:
        yes_no = parse_yes_no(answer)
        if yes_no is not None and yes_no in options:
            return yes_no

    # 2b. Pressupost obert ("no limit", "I don't have a budget"...) -> any.
    #     Va ABANS dels sinònims: "I don't have a budget" conté "budget".
    if answer_key == "budget_method" and "any" in options and is_open_budget_answer(answer):
        return "any"

    # 3. Sinònims específics del camp (type, budget_method, i paraules clau de
    #    terrace/elevator/commercial_use com "balcony", "lift", "shop"...)
    synonyms = SYNONYMS.get(answer_key, {})
    tokens = {t.lower() for t in nltk.word_tokenize(answer)}
    for option, words in synonyms.items():
        if option in options and tokens & {w.lower() for w in words}:
            return option

    # 4. Coincidència aproximada (només on té sentit, p. ex. location)
    if answer_key in FUZZY_FIELDS:
        fuzzy = fuzzy_match(answer, options)
        if fuzzy is not None:
            return fuzzy

    return None


def check_numeric_ranges_against_data(house_data):
    """Compara NUMERIC_RANGES amb els valors reals de les cases i avisa si alguna
    casa queda fora (així es veu de seguida si cal ajustar els rangs)."""
    for key in ("bedrooms", "bathrooms", "square_meters", "floor"):
        values = [numeric_value_from_text(h.get(key)) for h in house_data["houses"]]
        values = [v for v in values if v is not None]
        if not values:
            continue
        low, high = NUMERIC_RANGES[key]
        status = "OK" if low <= min(values) and max(values) <= high else "⚠ REVISAR"
        print(f"{key:14s} cases: {min(values):g}–{max(values):g} | rang permès: {low:g}–{high:g}  {status}")


check_numeric_ranges_against_data(data)


bedrooms       cases: 1–5 | rang permès: 1–5  OK
bathrooms      cases: 1–3 | rang permès: 1–3  OK
square_meters  cases: 45–150 | rang permès: 30–200  OK
floor          cases: 0–8 | rang permès: 0–10  OK


## 7.6 Flux de la conversa i classe `DialogEngine`

Aquesta cel·la conté la part conversacional, dividida en tres fases (vegeu el diagrama de flux del diàleg):

1. **Fase 1 · preguntes**: `QUESTION_ORDER` i `build_question_flow` decideixen quines preguntes es fan segons les respostes anteriors (compra o lloguer, mètode de pressupost, ús comercial).
2. **Fase 2 · resultats**: es mostren les cases exactes i, després, les alternatives.
3. **Fase 3 · relaxar requisits**: l'usuari pot rebaixar un requisit cada vegada i tornar a cercar.

Al davant hi ha les funcions sense estat (missatges de confirmació, interpretació de les comandes de les llistes). Després, la classe `DialogEngine` guarda l'estat de la conversa (pregunta actual, preferències, historial per a `back`, mode) i implementa les tres fases. `handle_answer` és l'entrada principal de cada missatge de l'usuari.

In [8]:
import random

# ---------------------------------------------------------------------
# Funcions "sense estat": donada una entrada, sempre calculen el mateix
# resultat. No necessiten ser mètodes de cap classe.
# ---------------------------------------------------------------------

# FASE 1 · Ordre "lògic" de les preguntes. Algunes són condicionals:
#  - budget_method    -> només si type == 'rent'
#  - monthly_income   -> només si type == 'rent' i budget_method == 'income'
#  - price            -> si type == 'sale', o si type == 'rent' i budget_method == 'budget'
#  - commercial_floor -> només si commercial_use == 'Yes' (proposem planta baixa)
#  - floor            -> sempre, tret que commercial_use == 'Yes' i l'usuari accepti la
#                        planta baixa. Va just després del bloc d'ús comercial, de manera
#                        que totes les branques convergeixen en aquesta pregunta.
QUESTION_ORDER = [
    "type", "budget_method", "monthly_income", "price",
    "commercial_use", "commercial_floor", "floor",
    "bedrooms", "bathrooms", "square_meters", "location", "terrace", "elevator",
]


def get_question(answer_key):
    for question in data["questions"]:
        if question["answer_key"] == answer_key:
            return question
    raise KeyError(f"No hi ha cap pregunta amb answer_key={answer_key!r}")


def question_text(question):
    variants = question.get("question_variants", [question["question"]])
    return random.choice(variants)


def question_options(question):
    key = question["answer_key"]
    if key in CUSTOM_OPTIONS:
        return CUSTOM_OPTIONS[key]
    return sorted(available_options.get(key, set()))


def wants_ground_floor_only(preferences):
    """True si és per a ús comercial I l'usuari ha acceptat la planta baixa."""
    return (preferences.get("commercial_use") == "Yes"
            and preferences.get("commercial_floor") == "Yes")


def build_question_flow(user_preferences):
    """Ordre de preguntes. `budget_method`/`monthly_income`/`price` depenen
    de com s'hagi respost a preguntes anteriors (veure QUESTION_ORDER)."""
    flow = []
    house_type = user_preferences.get("type")
    budget_method = user_preferences.get("budget_method")

    for key in QUESTION_ORDER:
        if key == "budget_method" and house_type != "rent":
            continue
        if key == "monthly_income":
            if house_type != "rent" or budget_method != "income":
                continue
        if key == "price":
            if house_type == "rent" and budget_method != "budget":
                continue
        if key == "commercial_floor" and user_preferences.get("commercial_use") != "Yes":
            continue
        if key == "floor" and wants_ground_floor_only(user_preferences):
            continue
        flow.append(get_question(key))
    return flow


# Respondre a aquestes preguntes pot canviar quines preguntes vénen després,
# així que cal reconstruir el flow just després de respondre-les
RECOMPUTE_FLOW_ON = {"type", "budget_method", "commercial_use", "commercial_floor"}


# Frases curtes de confirmació, variades perquè no soni sempre igual
CONFIRMATION_TEMPLATES = [
    "Got it — {value}.",
    "Perfect, noted: {value}.",
    "Great, {value} it is.",
    "Understood, {value}.",
]

# Camps numèrics per als quals afegim la unitat a la confirmació
# ("Got it — 3 bedrooms." en lloc de "Got it — 3.")
CONFIRMATION_UNITS = {
    "bedrooms": "bedroom(s)",
    "bathrooms": "bathroom(s)",
    "square_meters": "m²",
    "floor": "floor",
}


def confirmation_message(answer_key, value, uncertain=False):
    if uncertain:
        label = "the budget" if answer_key == "budget_method" else answer_key.replace("_", " ")
        return f"No worries, I'll mark {label} as 'any'."
    if answer_key == "budget_method" and value == "any":
        return "No budget limit, then — I won't filter by price."
    if answer_key == "commercial_floor":
        if value == "Yes":
            return "Perfect, I'll look for ground-floor properties."
        return "No problem."
    if value == "any":
        text = f"any {answer_key.replace('_', ' ')}"
    elif answer_key in CONFIRMATION_UNITS:
        text = f"{value:g} {CONFIRMATION_UNITS[answer_key]}" if isinstance(value, float) else f"{value} {CONFIRMATION_UNITS[answer_key]}"
    else:
        text = str(value)
    return random.choice(CONFIRMATION_TEMPLATES).format(value=text)



In [9]:

# ---------------------------------------------------------------------
# FASES 2 i 3 · Comandes que l'usuari pot escriure davant les llistes de
# cases i el menú de relaxació. Es busquen paraules soltes (per token) o
# frases curtes (per substring), igual que la resta de sinònims.
# ---------------------------------------------------------------------
RELAX_WORDS = {"relax", "loosen", "relajar", "flexibilize"}
RELAX_PHRASES = {"change a requirement", "change requirement", "drop a requirement",
                 "drop a filter", "less strict", "be more flexible"}

# "alternatives" a la llista d'exactes; "more" a la d'alternatives (però les
# dues paraules valen a les dues llistes)
ALTERNATIVES_WORDS = {"alternatives", "alternative", "alternativas", "alternativa",
                      "more", "next", "other", "others", "mas", "más"}

FINISH_WORDS = {"done", "finish", "finished", "stop", "bye", "goodbye", "enough", "cancel"}
FINISH_PHRASES = {"that's all", "thats all", "that is all", "i'm done", "im done",
                  "nothing else", "all good", "ya está", "ya esta"}

# Paraules clau per triar un requisit al menú de relaxació sense escriure el número
RELAX_FIELD_KEYWORDS = {
    "price": {"budget", "price", "cost", "spend", "money"},
    "bedrooms": {"bedroom", "bedrooms"},
    "bathrooms": {"bathroom", "bathrooms"},
    "location": {"location", "city", "neighborhood", "neighbourhood", "area", "zone"},
    "square_meters": {"size", "meters", "sqm", "space", "square"},
    "floor": {"floor", "storey"},
    "terrace": {"terrace", "balcony"},
    "elevator": {"elevator", "lift"},
    "commercial_floor": {"ground"},
}


def is_finish_answer(answer):
    """'no', 'done', 'that's all'... (acabar o cancel·lar)."""
    text = normalize_text(answer)
    if is_no_answer(text):
        return True
    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    return bool(tokens & FINISH_WORDS) or any(phrase in text for phrase in FINISH_PHRASES)


def parse_list_index(answer, count):
    """Converteix una selecció de llista (dígits, cardinals o ordinals) en índex."""
    text = normalize_text(answer)
    # "the second one" / "the 3rd one": traiem el "one" que va darrere d'un ordinal,
    # si no alpha2digit el converteix en un segon número ("2nd 1") i es rebutja
    ordinals = "|".join(ORDINAL_WORDS)
    text = re.sub(rf"\b({ordinals}|\d+(?:st|nd|rd|th))\s+one\b", r"\1", text)
    text = alpha2digit(text, "en", threshold=0)
    text = re.sub(r"\b(\d+)(?:st|nd|rd|th)\b", r"\1", text)
    tokens = text.split()
    text = " ".join(str(ORDINAL_WORDS.get(token, token)) for token in tokens)
    numbers = re.findall(r"\d+", text)
    if len(numbers) != 1:
        return None
    index = int(numbers[0])
    return index if 1 <= index <= count else None


def parse_results_command(answer):
    """Interpreta l'entrada davant d'una llista de cases. Retorna una tupla
    (comanda, número): ("relax", None), ("alternatives", None), ("finish", None),
    ("number", n), o None si no s'entén. Les comandes explícites van abans que el
    'no': "no, show me other options" és una petició d'alternatives."""
    text = normalize_text(answer)
    tokens = {t.lower() for t in nltk.word_tokenize(text)}

    if tokens & RELAX_WORDS or any(phrase in text for phrase in RELAX_PHRASES):
        return ("relax", None)
    if tokens & ALTERNATIVES_WORDS:
        return ("alternatives", None)
    if is_finish_answer(text):
        return ("finish", None)

    index = parse_list_index(text, 10**9)
    if index is not None:
        return ("number", index)
    return None


def parse_relaxation_field(answer, options):
    """Tria un requisit del menú de relaxació pel seu número o per paraula clau
    ("the budget", "terrace"...). Retorna el camp, o None si no és vàlid."""
    text = normalize_text(answer)
    index = parse_list_index(text, len(options))
    if index is not None:
        return options[index - 1]

    tokens = {t.lower() for t in nltk.word_tokenize(text)}
    matches = [field for field in options if tokens & RELAX_FIELD_KEYWORDS[field]]
    return matches[0] if len(matches) == 1 else None


def number_range_text(count):
    return "1" if count == 1 else f"a number between 1 and {count}"



In [10]:

# ---------------------------------------------------------------------
# DialogEngine: aquí sí que hi ha estat real (quina pregunta toca, què s'ha
# respost, l'historial per al 'back'...), així que té sentit encapsular-ho
# en una classe en lloc del diccionari global `dialog_state` d'abans.
# Cada trucada a chatbot_v7() crea una instància nova, en lloc de "resetejar"
# un global a mà.
#
# Modes (self.mode):
#   "asking"       fase 1 · preguntes
#   "exact_list"   fase 2 · llista de cases exactes (0 desajustos)
#   "alt_list"     fase 2 · alternatives (cases amb desajustos)
#   "relax_choice" fase 3 · quin requisit es relaxa
#   "relax_more"   fase 3 · "vols relaxar-ne algun més?"
#   "finished"     el diàleg ha acabat
# ---------------------------------------------------------------------
class DialogEngine:
    def __init__(self, data, debug=False):
        self.data = data
        self.flow = None
        self.step = 0
        self.preferences = {}
        self.history = []              # pila de (flow, step, preferences) per a 'back'
        self.mode = "asking"
        self.debug = debug
        self.awaiting_back_confirmation = False
        self.awaiting_exit_confirmation = False
        self.last_prompt = None        # últim missatge-pregunta (per tornar-lo a mostrar)

        # Resultat de l'última cerca (fase 2)
        self.scored = []               # [(casa, desajustos)] ordenat
        self.exact_houses = []
        self.alternatives = []         # [(casa, desajustos)] ordenat, sense cases bloquejades
        self.alt_shown = 0             # quantes alternatives s'han ensenyat ja
        self.alt_page = []             # les alternatives que es veuen ara mateix
        self.alt_header = ""
        self.last_list = None          # "exact_list" | "alt_list" | None: llista a la qual tornar

        # Menú de relaxació (fase 3)
        self.relax_options = []
        self.pending_changes = False   # s'ha relaxat algun requisit des de l'última cerca?

    def say(self, message):
        """Com bubble_agent, però recorda l'últim missatge que espera resposta
        (així, si l'usuari cancel·la la sortida, el podem repetir)."""
        self.last_prompt = message
        bubble_agent(message)

    # -- FASE 1 · preguntes --------------------------------------------
    def start(self):
        bubble_agent(self.data["start_message"])
        self.flow = build_question_flow(self.preferences)
        self.ask_current_question()

    def current_question(self):
        if self.flow is None or self.step >= len(self.flow):
            return None
        return self.flow[self.step]

    def ask_current_question(self):
        question = self.current_question()
        if question is None:
            return

        # Ús comercial + ha rebutjat la planta baixa: preguntem la planta mínima normalment
        if question["answer_key"] == "floor" and self.preferences.get("commercial_use") == "Yes":
            bubble_agent("Alright, then let's choose the floor yourself.")

        self.say(question_text(question))

    def end_conversation(self):
        self.mode = "finished"
        bubble_agent(self.data["end_message"])
        text_box.disabled = True
        send_btn.disabled = True

    def build_summary(self, searching_text="searching..."):
        """Frase curta amb el resum de què s'ha triat, abans de cercar."""
        preferences = self.preferences
        parts = []
        house_type = preferences.get("type")
        if house_type == "sale":
            parts.append("Buying")
        elif house_type == "rent":
            parts.append("Renting")

        for key in ["bedrooms", "bathrooms"]:
            value = preferences.get(key)
            if value not in (None, "any"):
                unit = CONFIRMATION_UNITS.get(key, key)
                parts.append(f"{value:g} {unit}" if isinstance(value, float) else f"{value} {unit}")

        sqm = preferences.get("square_meters")
        if sqm not in (None, "any"):
            parts.append(f"~{sqm:g} m²" if isinstance(sqm, float) else f"~{sqm} m²")

        location = preferences.get("location")
        if location not in (None, "any"):
            parts.append(str(location))

        price = preferences.get("price")
        if price not in (None, "any"):
            parts.append(f"up to {price:g}€" if isinstance(price, float) else f"up to {price}€")
        if price == "any" or preferences.get("budget_method") == "any":
            parts.append("no budget limit")

        # Els ingressos només es mencionen si el pressupost surt d'ells (si s'ha
        # relaxat, ja hi ha un preu explícit que mana)
        income = preferences.get("monthly_income")
        if income not in (None, "any") and price in (None, "any"):
            parts.append(f"budget based on {income:g}€/month income" if isinstance(income, float) else f"budget based on {income}€/month income")

        if preferences.get("commercial_use") == "Yes":
            parts.append("commercial use")
        if wants_ground_floor_only(preferences):
            parts.append("ground floor")
        floor = _pref_value(preferences, "floor")
        if floor is not None and float(floor) > 0:
            parts.append(f"floor {format_number(float(floor))} or higher")
        if preferences.get("terrace") == "Yes":
            parts.append("with terrace")
        if preferences.get("elevator") == "Yes":
            parts.append("with elevator")

        if not parts:
            return searching_text.capitalize()
        return ", ".join(parts) + " — " + searching_text

    # -- FASE 2 · puntuació i presentació de resultats --------------------
    def search(self, searching_text="searching..."):
        """Resum + puntuació de totes les cases. Si n'hi ha d'exactes (0 desajustos)
        es mostren totes; si no, es passa directament a les alternatives."""
        bubble_agent(self.build_summary(searching_text))

        self.scored = score_houses(self.data, self.preferences)
        self.exact_houses, self.alternatives = split_scored(self.scored)
        self.alt_shown = 0
        self.alt_page = []
        self.last_list = None
        self.pending_changes = False

        if self.debug:
            debug_message(f"Exact houses: {len(self.exact_houses)} | "
                          f"Alternatives: {len(self.alternatives)} | "
                          f"Preferences: {self.preferences}")

        if self.exact_houses:
            self.show_exact_houses()
            return

        bubble_agent("I couldn't find a house that matches everything you asked for.")
        self.show_alternatives()

    def results_options_text(self):
        """Què pot escriure l'usuari ara mateix (depèn de la llista que veu)."""
        options = ["a number to see the full details"]
        if self.mode == "exact_list":
            if self.alternatives:
                options.append("'alternatives' to see the closest houses that don't match everything")
        elif self.alt_shown < len(self.alternatives):
            options.append("'more' to see more alternatives")
        options.append("'relax' to relax a requirement")
        options.append("'done' to finish")
        return "Type " + ", ".join(options[:-1]) + ", or " + options[-1] + "."

    def show_exact_houses(self):
        self.mode = "exact_list"
        self.last_list = "exact_list"

        lines = [f"I found {len(self.exact_houses)} house(s) that match everything you asked for:"]
        for i, house in enumerate(self.exact_houses, start=1):
            lines.append(f"{i}. {house_one_line(house)}")
        lines.append("Do you like any of these, or would you rather see alternatives?")
        lines.append(self.results_options_text())
        self.say("\n".join(lines))

    def show_alternatives(self):
        """Ensenya les següents MAX_ALTERNATIVES_PER_PAGE millors cases amb desajustos."""
        page = self.alternatives[self.alt_shown:self.alt_shown + MAX_ALTERNATIVES_PER_PAGE]

        if not page:
            if self.alt_page:
                # Ja s'han ensenyat totes: avisem i tornem a la llista
                bubble_agent("There are no more alternatives.")
                self.say_alternatives_page()
            elif self.exact_houses:
                bubble_agent("I don't have any other houses that come close.")
                self.show_exact_houses()
            else:
                bubble_agent("I'm sorry, I couldn't find any houses that fit even roughly.")
                self.start_relaxation()
            return

        self.alt_header = ("These are the closest houses (they don't match everything):"
                           if self.alt_shown == 0 else "Here are some more alternatives:")
        self.alt_page = page
        self.alt_shown += len(page)
        self.say_alternatives_page()

    def say_alternatives_page(self):
        self.mode = "alt_list"
        self.last_list = "alt_list"

        lines = [self.alt_header]
        for i, (house, mismatches) in enumerate(self.alt_page, start=1):
            differences = ", ".join(m.text for m in mismatches)
            lines.append(f"{i}. {house_one_line(house)} — differs in: {differences}")
        lines.append("What would you like to do?")
        lines.append(self.results_options_text())
        self.say("\n".join(lines))

    def return_to_results(self):
        """Torna a la llista que es veia abans d'entrar a la relaxació."""
        if self.last_list == "exact_list":
            self.show_exact_houses()
        elif self.last_list == "alt_list":
            self.say_alternatives_page()
        else:
            bubble_agent("Feel free to start over with different preferences.")
            self.end_conversation()

    def handle_results_answer(self, answer):
        command, number = parse_results_command(answer) or (None, None)

        if command is None:
            bubble_agent("Sorry, I didn't understand that. " + self.results_options_text())
            return

        if command == "finish":
            bubble_agent("Ok, no worries.")
            self.end_conversation()
        elif command == "relax":
            self.start_relaxation()
        elif command == "alternatives":
            self.show_alternatives()
        else:  # "number": detall complet i tornem a la mateixa llista
            houses = self.exact_houses if self.mode == "exact_list" else [h for h, _ in self.alt_page]
            if not 1 <= number <= len(houses):
                bubble_agent("That number isn't on the list. " + self.results_options_text())
                return
            bubble_agent(house_summary(houses[number - 1]))
            if self.mode == "exact_list":
                self.show_exact_houses()
            else:
                self.say_alternatives_page()

    # -- FASE 3 · relaxar requisits -----------------------------------------
    def start_relaxation(self):
        """Llista els requisits actius amb les cases extra que apareixerien."""
        self.scored = score_houses(self.data, self.preferences)
        self.relax_options = active_requirements(self.preferences)

        if not self.relax_options:
            bubble_agent("There's nothing left I can relax.")
            self.return_to_results()
            return

        self.mode = "relax_choice"
        lines = ["Which requirement would you like to relax?"]
        for i, field in enumerate(self.relax_options, start=1):
            extra = format_extra_houses(unlocked_by_relaxing(self.scored, self.preferences, field))
            lines.append(f"{i}. {describe_relaxation(field, self.preferences)} — {extra}")
        lines.append("Type the number (or 'no' to keep everything as it is).")
        self.say("\n".join(lines))

    def handle_relax_choice(self, answer):
        if is_finish_answer(answer):
            bubble_agent("Ok, I'll keep the requirements as they are.")
            if self.pending_changes:
                self.search("searching again...")   # s'ha relaxat alguna cosa: cal tornar a puntuar
            else:
                self.return_to_results()            # res no ha canviat: tornem a la llista de sempre
            return

        field = parse_relaxation_field(answer, self.relax_options)
        if field is None:
            bubble_agent(f"Please type {number_range_text(len(self.relax_options))}, or 'no'.")
            return

        apply_relaxation(self.preferences, field)
        self.pending_changes = True
        bubble_agent(relaxation_confirmation(field, self.preferences))

        if not active_requirements(self.preferences):
            bubble_agent("There's nothing else to relax.")
            self.search("searching again...")
            return

        self.mode = "relax_more"
        self.say("Do you want to relax anything else? (yes/no)")

    def handle_relax_more(self, answer):
        yes_no = parse_yes_no(answer)
        if yes_no == "Yes":
            self.start_relaxation()
        elif yes_no == "No" or is_finish_answer(answer):
            self.search("searching again...")
        else:
            bubble_agent("Sorry, I didn't get that. Do you want to relax anything else? (yes/no)")

    # -- navegació -------------------------------------------------------
    def go_back(self):
        """Torna a la pregunta anterior sense reiniciar tot el diàleg."""
        if not self.history:
            bubble_agent("We're already at the first question, there's nothing to go back to.")
            self.ask_current_question()
            return

        self.flow, self.step, self.preferences = self.history.pop()
        bubble_agent("Sure, let's go back.")
        self.ask_current_question()

    # -- sortir del xat (amb confirmació) --------------------------------
    def request_exit(self):
        self.awaiting_exit_confirmation = True
        bubble_agent("Are you sure you want to leave the chat? (yes/no)")

    def handle_exit_confirmation(self, answer):
        # "q" / "quit" una segona vegada també compta com a "yes"
        yes_no = "Yes" if is_exit_command(answer) else parse_yes_no(answer)
        if yes_no == "Yes":
            self.awaiting_exit_confirmation = False
            self.end_conversation()
        elif yes_no == "No":
            self.awaiting_exit_confirmation = False
            bubble_agent("Ok, let's continue.")
            if self.last_prompt:
                bubble_agent(self.last_prompt)
        else:
            bubble_agent("Sorry, I didn't get that. Do you want to leave the chat? (yes/no)")

    # -- entrada principal -------------------------------------------------
    def handle_answer(self, _=None):
        if self.mode == "finished":
            return

        raw_answer = text_box.value.strip()
        if raw_answer == "":
            return

        if self.mode == "asking" and self.current_question() is None:
            return

        # Un únic punt de normalització per a TOT el diàleg (minúscules,
        # apòstrofs, abreviatures). L'usuari veu el seu text original.
        answer = normalize_text(raw_answer)

        bubble_user(raw_answer)
        text_box.value = ""

        # Sortir: demanem confirmació en qualsevol fase
        if self.awaiting_exit_confirmation:
            self.handle_exit_confirmation(answer)
            return
        if is_exit_command(answer):
            self.request_exit()
            return

        # Fases 2 i 3: resultats i relaxació ('back' només funciona a la fase 1)
        if self.mode in ("exact_list", "alt_list"):
            self.handle_results_answer(answer)
            return
        if self.mode == "relax_choice":
            self.handle_relax_choice(answer)
            return
        if self.mode == "relax_more":
            self.handle_relax_more(answer)
            return

        question = self.current_question()

        # Resposta a "vols tornar enrere?" (després de detectar un "m'he equivocat")
        if self.awaiting_back_confirmation:
            self.awaiting_back_confirmation = False
            yes_no = "Yes" if is_back_command(answer) else parse_yes_no(answer)
            if yes_no == "Yes":
                self.go_back()
            elif yes_no == "No":
                bubble_agent("Ok, let's continue.")
                self.ask_current_question()
            else:
                self.say("Sorry, I didn't get that. Do you want to go back to the previous question? (yes/no)")
                self.awaiting_back_confirmation = True
            return

        # Petició de tornar enrere ("go back", "previous question"...) o "m'he
        # equivocat": SEMPRE demanem confirmació abans de tornar enrere.
        wants_back = is_back_command(answer)
        if wants_back or is_mistake_comment(answer):
            if not self.history:
                self.go_back()  # ja som a la primera pregunta (ho explica go_back)
            else:
                intro = "" if wants_back else "It sounds like you made a mistake. "
                self.say(intro + "Do you want to go back to the previous question? (yes/no)")
                self.awaiting_back_confirmation = True
            return

        answer_key = question["answer_key"]
        uncertain = False

        if is_uncertain_answer(answer):
            # "I don't know" no és un "No": com que no n'està segur, ho apuntem com a any
            value = "any"
            uncertain = True
        elif is_any_command(answer):
            value = "any"
        elif question["type"] == "numerical":
            value, error = validate_numeric_answer(answer, answer_key)
            if error is not None:
                bubble_agent(error)
                return
        else:
            value = validate_multichoice_answer(answer, question_options(question), answer_key)
            if value is None:
                bubble_agent("Sorry, I didn't recognise that option. "
                              f"Options: {', '.join(question_options(question))}. "
                              "Or type 'any' if it doesn't matter to you.")
                return

        # Guardem un "snapshot" ABANS d'aplicar la resposta, per poder desfer-la amb 'back'
        self.history.append((
            list(self.flow) if self.flow is not None else None,
            self.step,
            dict(self.preferences),
        ))

        self.preferences[answer_key] = value

        # Pressupost obert: no hi ha límit, així que no cal preguntar ni ingressos ni preu
        if answer_key == "budget_method" and value == "any":
            self.preferences["price"] = "any"

        bubble_agent(confirmation_message(answer_key, value, uncertain))

        if self.debug:
            debug_message(
                f"Pregunta: {answer_key} | "
                f"Respuesta guardada: {value} | "
                f"Preferencias actuales: {self.preferences}"
            )

        if answer_key in RECOMPUTE_FLOW_ON:
            self.flow = build_question_flow(self.preferences)

        self.step += 1

        if self.current_question() is not None:
            self.ask_current_question()
        else:
            self.search()



In [11]:

# La instància "viva" de la conversa actual. chatbot_v7() la torna a crear
# cada vegada que es (re)inicia el xat.
dialog_engine = None


def handle_answer(_=None):
    """Funció embolcall registrada als widgets (send_btn.on_click / text_box.on_submit).
    Delega sempre a la instància actual de DialogEngine, així no cal tornar a
    registrar els callbacks cada vegada que es crea una conversa nova."""
    if dialog_engine is not None:
        dialog_engine.handle_answer()

## 7.7 Puntuació de cases i relaxació de requisits

Lògica de cerca, que no depèn de cap estat. `score_houses` recorre les cases una sola vegada i, per a cadascuna, `house_mismatches` calcula la llista de desajustos amb els requisits de l'usuari (cada un amb una gravetat entre 0 i 1). Les cases amb 0 desajustos són les exactes; la resta s'ordenen per nombre de desajustos, gravetat total i id. Aquí també hi ha les constants ajustables (marges de m², percentatge de relaxació del pressupost, pesos de gravetat) i les funcions de la fase 3: quins requisits es poden relaxar, quantes cases més apareixerien en relaxar-ne un i com s'aplica el canvi.

In [12]:
from collections import namedtuple

# ---------------------------------------------------------------------
# Cerca per puntuació (fase 2) i relaxació de requisits (fase 3)
#
# Una sola funció (score_houses) puntua TOTES les cases: cada casa té una
# llista de "desajustos" (els requisits de l'usuari que no compleix).
#   - 0 desajustos        -> casa exacta
#   - 1 o més desajustos  -> alternativa (ordenades de millor a pitjor)
# No hi ha una funció de cerca exacta a part: així els criteris només
# estan escrits UNA vegada.
# ---------------------------------------------------------------------

# Marge de metres quadrats: no simètric, perquè preferim oferir cases una mica
# més GRANS que el que es demana abans que més petites ("millor passar-se que quedar-se curt").
SQM_MARGIN_BELOW = 10   # m2 que pot faltar (casa més petita del que es demana)
SQM_MARGIN_ABOVE = 25   # m2 que pot sobrar (casa més gran del que es demana)

# Percentatge en què puja el topall de pressupost cada vegada que l'usuari
# demana relaxar-lo. La cerca sempre fa servir el pressupost com a topall estricte.
BUDGET_RELAXATION_PERCENT = 0.10  # 10%

# Cada desajust puntua entre 0 (petit) i 1 (greu). Les cases s'ordenen primer
# pel NOMBRE de desajustos i, a igualtat, per la gravetat total.
YES_NO_MISMATCH_SEVERITY = 0.3   # terrassa / ascensor que falten
LOCATION_MISMATCH_SEVERITY = 0.6
GROUND_FLOOR_MISMATCH_SEVERITY = 1.0

# Cases que es mostren cada vegada a la llista d'alternatives
MAX_ALTERNATIVES_PER_PAGE = 3

Mismatch = namedtuple("Mismatch", ["field", "text", "severity"])

# Requisits "durs" que es poden relaxar: una casa que no els compleix NO entra
# a cap llista (ni exactes ni alternatives), però sí que compta per saber
# quantes cases apareixerien si l'usuari els relaxa.
# (type i commercial_use = Yes també són durs, però no es relaxen mai: es
# filtren abans, a passes_hard_requirements.)
BLOCKING_FIELDS = {"commercial_floor"}


def _pref_value(preferences, key):
    """Retorna el valor d'una preferència, o None si no existeix o és 'any'."""
    value = preferences.get(key)
    return None if value in (None, "any") else value


def effective_budget_cap(preferences):
    """Calcula el topall de despesa efectiu, tant si ve d'un preu màxim explícit
    com si ve dels ingressos (regla del 35%). Si hi ha un 'price' explícit
    (per exemple perquè s'ha relaxat), aquest sempre té prioritat sobre el
    càlcul a partir dels ingressos."""
    price_value = _pref_value(preferences, "price")
    if price_value is not None:
        return numeric_value_from_text(price_value)

    income_value = _pref_value(preferences, "monthly_income")
    if income_value is not None:
        income = numeric_value_from_text(income_value)
        return income * 0.35 if income is not None else None

    return None


def format_number(value):
    """1200.0 -> '1200', 1098900 -> '1098900', 2.5 -> '2.5' (sense notació científica)."""
    return f"{value:.2f}".rstrip("0").rstrip(".")


def format_price(house):
    """800 (rent) -> '800€/month'; 250k (sale) -> '250k€'."""
    suffix = "/month" if house["type"] == "rent" else ""
    return f"{house['price']}€{suffix}"


def house_one_line(house):
    """Una línia per casa: id, zona i preu (és el que es veu a les llistes)."""
    return f"House {house['id']} · {house['location']} · {format_price(house)}"


def house_summary(house):
    """Detall complet d'una casa (només es mostra quan l'usuari la tria)."""
    return (
        f"{house_one_line(house)} "
        f"— {house['bedrooms']} bedrooms, "
        f"{house['bathrooms']} bathrooms, {house['square_meters']} m², "
        f"floor {house['floor']}, elevator: {house['elevator']}, "
        f"terrace: {house['terrace']}, commercial use: {house['commercial_use']}"
    )


# ---------------------------------------------------------------------
# Puntuació
# ---------------------------------------------------------------------
def passes_hard_requirements(house, preferences):
    """Requisits que mai es negocien: tipus d'operació i ús comercial = Yes."""
    requested_type = _pref_value(preferences, "type")
    if requested_type is not None and house.get("type") != requested_type:
        return False
    # Només "Yes" restringeix: si no l'usarà com a negoci, tant li fa que
    # la casa ho permeti o no.
    if preferences.get("commercial_use") == "Yes" and house.get("commercial_use") != "Yes":
        return False
    return True


def house_mismatches(house, preferences):
    """Llista de Mismatch(camp, text, gravetat) amb els requisits que la casa no compleix."""
    mismatches = []

    cap = effective_budget_cap(preferences)
    price = numeric_value_from_text(house["price"])
    if cap is not None and price is not None and price > cap:
        mismatches.append(Mismatch(
            "price",
            f"price ({format_price(house)} vs up to {format_number(cap)}€)",
            min(1.0, (price - cap) / cap),
        ))

    sqm_value = _pref_value(preferences, "square_meters")
    if sqm_value is not None:
        requested = numeric_value_from_text(sqm_value)
        actual = numeric_value_from_text(house["square_meters"])
        diff = actual - requested
        excess = -diff - SQM_MARGIN_BELOW if diff < -SQM_MARGIN_BELOW else max(0, diff - SQM_MARGIN_ABOVE)
        if excess > 0:
            mismatches.append(Mismatch(
                "square_meters",
                f"size ({format_number(actual)} vs ~{format_number(requested)} m²)",
                min(1.0, excess / SQM_MARGIN_ABOVE),
            ))

    # Planta mínima: no s'aplica si l'usuari ha acceptat la planta baixa
    floor_value = _pref_value(preferences, "floor")
    if floor_value is not None and not wants_ground_floor_only(preferences):
        requested = float(floor_value)
        actual = numeric_value_from_text(house["floor"])
        if actual < requested:
            mismatches.append(Mismatch(
                "floor",
                f"floor ({format_number(actual)} vs {format_number(requested)}+)",
                min(1.0, (requested - actual) / max(requested, 1)),
            ))

    for field in ("bedrooms", "bathrooms"):
        value = _pref_value(preferences, field)
        if value is not None:
            actual = numeric_value_from_text(house[field])
            if actual != float(value):
                mismatches.append(Mismatch(
                    field,
                    f"{field} ({format_number(actual)} vs {format_number(float(value))})",
                    min(1.0, abs(actual - float(value)) / 2),
                ))

    location = _pref_value(preferences, "location")
    if location is not None and house["location"] != location:
        mismatches.append(Mismatch(
            "location", f"location ({house['location']})", LOCATION_MISMATCH_SEVERITY,
        ))

    for field in ("terrace", "elevator"):
        if preferences.get(field) == "Yes" and house.get(field) != "Yes":
            mismatches.append(Mismatch(field, f"{field} (No)", YES_NO_MISMATCH_SEVERITY))

    # Planta baixa per a ús comercial (requisit dur, però relaxable)
    if wants_ground_floor_only(preferences) and numeric_value_from_text(house.get("floor", 0)) != 0:
        mismatches.append(Mismatch(
            "commercial_floor",
            f"floor ({house['floor']} vs ground floor)",
            GROUND_FLOOR_MISMATCH_SEVERITY,
        ))

    return mismatches


def score_houses(data, preferences):
    """UNA passada per la base de dades: [(casa, desajustos)] de totes les cases
    que compleixen type i commercial_use, ordenades de millor a pitjor
    (menys desajustos, menys gravetat total i, per desempatar, id)."""
    scored = []
    for house in data["houses"]:
        if not passes_hard_requirements(house, preferences):
            continue
        scored.append((house, house_mismatches(house, preferences)))

    scored.sort(key=lambda item: (len(item[1]), sum(m.severity for m in item[1]), int(item[0]["id"])))
    return scored


def is_blocked(mismatches):
    return any(m.field in BLOCKING_FIELDS for m in mismatches)


def split_scored(scored):
    """(cases exactes, alternatives). Les alternatives són (casa, desajustos) i
    no inclouen les cases bloquejades per un requisit dur."""
    exact = [house for house, mismatches in scored if not mismatches]
    alternatives = [(house, mismatches) for house, mismatches in scored
                    if mismatches and not is_blocked(mismatches)]
    return exact, alternatives


# ---------------------------------------------------------------------
# Relaxació de requisits
# ---------------------------------------------------------------------
# Requisits que l'usuari pot demanar relaxar, en l'ordre en què es mostren.
# NO hi són `type` (comprar/llogar) ni `commercial_use = Yes`.
RELAXABLE_FIELDS = [
    "price", "bedrooms", "bathrooms", "location", "square_meters",
    "floor", "terrace", "elevator", "commercial_floor",
]


def is_active_requirement(preferences, field):
    """True si l'usuari té aquest requisit actiu (és a dir, si té sentit relaxar-lo)."""
    if field == "price":
        return effective_budget_cap(preferences) is not None
    if field in ("terrace", "elevator"):
        return preferences.get(field) == "Yes"
    if field == "commercial_floor":
        return wants_ground_floor_only(preferences)
    if field == "floor":
        value = _pref_value(preferences, "floor")
        return value is not None and float(value) > 0   # "planta 0 o més" no filtra res
    return _pref_value(preferences, field) is not None


def active_requirements(preferences):
    return [field for field in RELAXABLE_FIELDS if is_active_requirement(preferences, field)]


def relaxed_budget_cap(preferences):
    """Topall actual + BUDGET_RELAXATION_PERCENT (vingui d'un preu directe o dels ingressos)."""
    return round(effective_budget_cap(preferences) * (1 + BUDGET_RELAXATION_PERCENT))


def apply_relaxation(preferences, field):
    """Relaxa UN requisit: el pressupost puja un %, la resta passen a 'any'."""
    if field == "price":
        preferences["price"] = relaxed_budget_cap(preferences)  # el preu explícit mana sobre els ingressos
    else:
        preferences[field] = "any"


def unlocked_by_relaxing(scored, preferences, field):
    """Quantes cases passarien a ser exactes si es relaxés `field`: les que NOMÉS
    fallen en aquest camp. Surt de la mateixa llista de desajustos, sense tornar a cercar."""
    new_cap = relaxed_budget_cap(preferences) if field == "price" else None
    count = 0
    for house, mismatches in scored:
        if {m.field for m in mismatches} != {field}:
            continue
        # Pujar el pressupost un % no desbloqueja QUALSEVOL casa més cara
        if field == "price" and numeric_value_from_text(house["price"]) > new_cap:
            continue
        count += 1
    return count


def format_extra_houses(count):
    if count == 0:
        return "no extra houses"
    return f"+{count} house" if count == 1 else f"+{count} houses"


def describe_relaxation(field, preferences):
    """Text d'una opció del menú de relaxació, adaptat a cada tipus de camp."""
    if field == "price":
        pct = int(round(BUDGET_RELAXATION_PERCENT * 100))
        return (f"budget: up to {format_number(effective_budget_cap(preferences))}€ "
                f"-> up to {format_number(relaxed_budget_cap(preferences))}€ (+{pct}%)")
    if field in ("terrace", "elevator"):
        return f"{field}: required -> not required"
    if field == "square_meters":
        return f"size: around {format_number(numeric_value_from_text(preferences[field]))} m² -> any size"
    if field == "floor":
        return f"minimum floor: {format_number(float(preferences[field]))} -> any floor"
    if field == "commercial_floor":
        return "ground floor only -> any floor"
    value = preferences[field]
    value = format_number(float(value)) if field in ("bedrooms", "bathrooms") else value
    return f"{field}: {value} -> any"


RELAXATION_LABELS = {
    "bedrooms": "the number of bedrooms",
    "bathrooms": "the number of bathrooms",
    "location": "the location",
    "square_meters": "the size",
    "floor": "the minimum floor",
    "terrace": "the terrace",
    "elevator": "the elevator",
    "commercial_floor": "the ground-floor requirement",
}


def relaxation_confirmation(field, preferences):
    """Frase de confirmació, a dir DESPRÉS d'haver aplicat apply_relaxation."""
    if field == "price":
        return f"Ok, the budget is now up to {format_number(effective_budget_cap(preferences))}€."
    return f"Ok, {RELAXATION_LABELS[field]} no longer matters."

## 7.8 Execució del xat

`chatbot_v7` reinicia la interfície i crea una instància nova de `DialogEngine`. La cel·la següent l'inicia amb el mode de depuració activat, i l'última registra els esdeveniments dels widgets (botó i tecla Enter) i mostra el xat.

In [13]:
def chatbot_v7(debug=False):
    global dialog_engine

    chat_history.clear()
    render_chat()

    text_box.disabled = False
    send_btn.disabled = False

    dialog_engine = DialogEngine(data, debug=debug)
    dialog_engine.start()


In [14]:
chatbot_v7(debug=True)  # debug=True per veure el flux de preguntes i respostes al xat

In [15]:
send_btn._click_handlers.callbacks.clear()
text_box._submission_callbacks.callbacks.clear()

send_btn.on_click(handle_answer)
text_box.on_submit(handle_answer)

display(chat_ui)

C:\Users\Nat\AppData\Local\Temp\ipykernel_27104\235671716.py:5: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_box.on_submit(handle_answer)
